# Telco Customer Churn — Feature Engineering

# 1. Objectives & Design Choices
This step focuses on transforming business insights identified during the EDA
into machine-learning-ready features, while ensuring interpretability and avoiding data leakage.

Key principles:
- Features must be business-driven and interpretable
- No target leakage
- Simple transformations over complex feature engineering

---

# 2. Data Loading & Target Definition
Load the raw dataset and create a clean binary target variable (`churn_flag`)
to be used for model training and evaluation.

---

# 3. Train/Test Split
Split the dataset into training and test sets before any preprocessing step
to prevent data leakage and ensure reliable model evaluation.

---

# 4. Feature Selection
Select a subset of features based on business relevance and insights from the EDA,
prioritizing contract type, tenure, pricing, and service-related variables.

---

# 5. Feature Encoding & Preprocessing
Apply appropriate preprocessing techniques:
- Scaling for numerical features
- One-hot encoding for categorical features

All transformations are defined using a scikit-learn `ColumnTransformer`
to ensure consistency and reusability.

---

# 6. Final Dataset Overview
Inspect the transformed training and test datasets
to validate dimensions and readiness for modeling.


In [68]:
import kagglehub            # KaggleHub: utility to programmatically download datasets/models from Kaggle
import os                   # OS-level utilities (paths, environment variables, directory handling)
import shutil               # High-level file operations (copy/move/delete folders/files)
import pandas as pd         # Core library for tabular data manipulation (DataFrames)
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer




In [69]:
# ============================================================
# Dataset acquisition & local storage
# Purpose:
# - Download the Telco Customer Churn dataset from Kaggle (only once)
# - Store it locally in data/raw/ for reproducibility
# ============================================================

destination_path = "../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv"

if not os.path.exists(destination_path):
    dataset_path = kagglehub.dataset_download("blastchar/telco-customer-churn")
    csv_files = [f for f in os.listdir(dataset_path) if f.endswith(".csv")]

    if len(csv_files) != 1:
        raise ValueError(f"Expected 1 CSV file, found {len(csv_files)}: {csv_files}")

    shutil.copy(os.path.join(dataset_path, csv_files[0]), destination_path)
    print("Dataset downloaded and copied to data/raw/")
else:
    print("Dataset already present in data/raw/")

df = pd.read_csv(destination_path)

# Stable binary target (do not overwrite df["Churn"])
df["churn_flag"] = df["Churn"].map({"Yes": 1, "No": 0}).astype(int)

df.head()


Dataset already present in data/raw/


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn,churn_flag
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No,0
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,No,No,No,One year,No,Mailed check,56.95,1889.5,No,0
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No,0
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1


In [70]:
# ============================================================
# Feature matrix (X) and target vector (y)
# Purpose: prepare data for supervised churn modeling
# ============================================================
# TotalCharges is sometimes stored as text with empty strings
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

# Define the feature matrix (X)
# - customerID is a unique identifier with no predictive value
# - Churn is the raw target label (kept only for EDA)
# - churn_flag is the binary target and must not appear in X
#   to avoid target leakage
X = df.drop(columns=["customerID", "Churn", "churn_flag"])

# Define the target vector (y)
# churn_flag = 1 indicates churn, 0 indicates retention
y = df["churn_flag"]




In [71]:
# ============================================================
# Train / test split
# Purpose: create independent training and testing datasets
# for reliable churn model evaluation
# ============================================================

# Split the dataset into training and test sets
# - test_size=0.2 reserves 20% of the data for evaluation
# - random_state ensures reproducibility
# - stratify=y preserves the churn / non-churn proportion
#   in both train and test sets (important for imbalanced data)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)



In [72]:
# ============================================================
# Feature categorization: categorical vs numerical variables
# Purpose: define feature types for appropriate preprocessing
# (encoding, scaling, imputation)
# ============================================================

# Categorical features
# These variables represent service types, contract modalities,
# or customer choices and will require encoding (e.g. One-Hot Encoding)
cat_features = [
    "Contract",
    "TechSupport",
    "OnlineSecurity",
    "InternetService",
    "PaperlessBilling",
    "PaymentMethod"
]

# Numerical features
# These variables are continuous or ordinal and can be
# scaled or imputed depending on the chosen model
num_features = ["tenure", "MonthlyCharges", "TotalCharges"]



In [ ]:
# ============================================================
# Preprocessing pipelines for numerical and categorical features
# Purpose: handle missing values and apply appropriate
# transformations in a clean, reusable, and leak-free way
# ============================================================

# Pipeline for numerical features
# - Median imputation is robust to outliers
# - Standardization is required for scale-sensitive models
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

# Pipeline for categorical features
# - Most frequent imputation preserves dominant categories
# - One-hot encoding converts categories into model-ready features
# - drop="first" avoids multicollinearity in linear models
# - handle_unknown="ignore" ensures robustness on unseen categories
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore"))
])

# Combine numerical and categorical pipelines
# using a ColumnTransformer for consistent preprocessing
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_features),
        ("cat", categorical_transformer, cat_features),
    ]
)



In [74]:
print("Train size:", X_train.shape, "Test size:", X_test.shape)
print("Churn rate train:", y_train.mean().round(4), "test:", y_test.mean().round(4))
print("Missing values in X_train:", X_train.isna().sum().sum())


Train size: (5634, 19) Test size: (1409, 19)
Churn rate train: 0.2654 test: 0.2654
Missing values in X_train: 8
